# Impureza GINI

### Contexto y Datos

* Una de las técnicas más populares son los "bosques aleatorios", los cuales consisten un conjunto de árboles de desición, los cuáles usan una métrica de decisión llamada GINI.
* GINI nos indica una probabiliad de clasificar erroneamente un dato.
* Estos datos corresponde a la cantidad de puntos promedio de 10 jugadores de basketball en su primer año.
* Alto indica que ese jugador tuvo un nivel de puntos altos en comparación al promedio de sus otros compañeros y caso contrario igual.
* Queremos determinar si estos jugadores conseguiran hacer una carrera en este equipo.

In [6]:
import pandas as pd

puntos_partidos = pd.Series([
    "alto", "bajo", "alto", "alto", "alto",
    "alto", "bajo", "alto", "alto", "bajo",
])

minutos_partidos = pd.Series([
    "alto", "alto", "bajo", "bajo", "bajo",
    "alto", "bajo", "bajo", "bajo", "alto",
])

rebotes_partidos = pd.Series([
    "alto", "bajo", "bajo", "alto", "bajo",
    "alto", "bajo", "alto", "bajo", "alto",
])

asistencia_partidos = pd.Series([
    "bajo", "bajo", "bajo", "bajo", "bajo",
    "bajo", "bajo", "bajo", "bajo", "bajo",
])

# 1: Veterano (Su carrera durará 5 años o más
clase = pd.Series([1,0,0,1,0,1,0,1,0,1])
datos = pd.DataFrame({
    "puntos": puntos_partidos,
    "minutos": minutos_partidos,
    "asistencias": asistencia_partidos,
    "rebotes": rebotes_partidos,
    "clase": clase
})

datos

,puntos,minutos,asistencias,rebotes,clase
0,alto,alto,bajo,alto,1
1,bajo,alto,bajo,bajo,0
2,alto,bajo,bajo,bajo,0
3,alto,bajo,bajo,alto,1
4,alto,bajo,bajo,bajo,0
5,alto,alto,bajo,alto,1
6,bajo,bajo,bajo,bajo,0
7,alto,bajo,bajo,alto,1
8,alto,bajo,bajo,bajo,0
9,bajo,alto,bajo,alto,1


### Interpretación de la Impureza GINI

* La impureza Gini es una medida que usan los árboles de decisión (por ejemplo, en scikit-learn) para evaluar qué tan “pura” o “mezclada” está una partición de datos respecto a sus clases.
$$
Gini = 1 - \sum_{i=1}^{K} p_i^2
$$
* Una partición es pura si todos los ejemplos pertenecen a la misma clase (impureza Gini = 0).
* Una partición es impura si hay mezcla de clases, y su valor máximo depende del número de clases.
* Supongamos un nodo de un árbol de decisión con 10 muestras
  - 4 son rojas (clase 1) --> P(rojo) = 40%
  - 6 son azules (clase 2) --> P(azul) = 60%
* Gini=1−(0.16+0.36)=1−0.52=0.48
* 0.0	Nodo puro (todas las muestras son de la misma clase).
* 0.5	Máxima mezcla en caso de 2 clases (50/50).
* <0.5	Hay cierta pureza, pero sigue habiendo mezcla.

***NOTAS***
* Podemos ver como `Rebotes` es que tiene una pureza mejor. Es decir podemos clasificar de una manera perfecta.
* En `asistencias` tiene un 0.5, además en la tabla podemos ver que ningun jugador tiene la categoría "alta", es como si falta el otro 50% de la clasificación, tiene el nivel máximo de impureza, por lo que esta columna no nos aporta nada.
* Por otro lado `puntos` y `minutos`, están cerca de la impureza máxima, entonces la mejor opción para clasificar sería por los `rebotes`, despues los minutos, puntos y al final las asistencias.

In [7]:
def impureza_gini(caracteristica, clase, datos):
    """str, str, DataFrame -> Floart"""
    atributo_clase = datos.groupby([caracteristica, clase])[clase].count()
    atributo = datos.groupby([caracteristica])[clase].count()
    procesados = pd.merge(atributo_clase,atributo, on=[caracteristica],
                         suffixes=('_individual', '_total'))
    procesados["combinacion"] = (procesados[clase+"_individual"]/
                                 procesados[clase+"_total"])**2
    gini_combinacion = 1 - procesados.groupby([caracteristica, clase+"_total"])["combinacion"].sum()
    gini_pesado = (gini_combinacion * atributo) / atributo.sum()
    return gini_pesado.sum()

print("Puntos -> impureza: %0.4f" % impureza_gini("puntos", "clase", datos))
print("Minutos -> impureza: %0.4f" % impureza_gini("minutos", "clase", datos))
print("Asistencias -> impureza: %0.4f" % impureza_gini("asistencias", "clase", datos))
print("Rebotes -> impureza: %0.4f" % impureza_gini("rebotes", "clase", datos))
datos

Puntos -> impureza: 0.4762
Minutos -> impureza: 0.4167
Asistencias -> impureza: 0.5000
Rebotes -> impureza: 0.0000


,puntos,minutos,asistencias,rebotes,clase
0,alto,alto,bajo,alto,1
1,bajo,alto,bajo,bajo,0
2,alto,bajo,bajo,bajo,0
3,alto,bajo,bajo,alto,1
4,alto,bajo,bajo,bajo,0
5,alto,alto,bajo,alto,1
6,bajo,bajo,bajo,bajo,0
7,alto,bajo,bajo,alto,1
8,alto,bajo,bajo,bajo,0
9,bajo,alto,bajo,alto,1


### Impureza del atributo 'puntos'

In [9]:
### Cuántos jugarores por categoría de puntos y por clase (novato o veterano)
puntos_clase = datos.groupby(["puntos", "clase"])["clase"].count()
print("\n\n JUGADORES POR PUNTOS Y CLASE \n\n", puntos_clase)
puntos = datos.groupby(["puntos"])["clase"].count()

### Cuantos jugadores de datos para procesamiento posteior
jugadores = pd.merge(
    puntos_clase,
    puntos,
    on=["puntos"],
    suffixes=('_indivual', '_total')
)
print("\n\n UNION \n\n", jugadores)



 JUGADORES POR PUNTOS Y CLASE 

 puntos  clase
alto    0        3
        1        4
bajo    0        2
        1        1
Name: clase, dtype: int64


 UNION 

         clase_indivual  clase_total
puntos                             
alto                 3            7
alto                 4            7
bajo                 2            3
bajo                 1            3


### Cálculo de la Impureza GINI para 'puntos'

In [ ]:
# Probabilidad para cada categoría de puntos con respecto a la clase
jugadores["combinaciones"] = (jugadores["clase_indivual"]/jugadores["clase_total"]**2)
print(jugadores)

# Impureza gini para cada combinación
gini_por_combinacion = 1 - jugadores